In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
# os.environ['GEMINI_API_KEY']=os.getenv("GEMINI_API_KEY")
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")
# print("GEMINI_API_KEY:", os.environ['GEMINI_API_KEY'])
os.environ["TAVILY_API_KEY"]=os.getenv("TRAVILY_API_KEY")

from langchain.chat_models import init_chat_model
# model = init_chat_model(model="llama-3.1-8b-instant", model_provider="groq",
#                             api_key=os.getenv("GROQ_API_KEY"))

model = init_chat_model(model="gemini-3-flash-preview", model_provider="google_genai",
                            api_key=os.getenv("GEMINI_API_KEY"))

In [2]:
model

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-3-flash-preview', temperature=1.0, client=<google.genai.client.Client object at 0x000002012E665B50>, default_metadata=(), model_kwargs={})

In [3]:
from langchain_tavily import TavilySearch

travilySearch_tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # include_favicon=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None,
    # country=None
)

In [ ]:
# Basic query
# travilySearch_tool.invoke({"query": "What happened at the last wimbledon"})

In [4]:
from langchain.tools import tool

@tool
def calculator_tool(expression: str) -> str:
    """A simple calculator tool to evaluate mathematical expressions."""
    try:
        result = str(eval(expression))
    except Exception as e:
        result = f"Error evaluating expression: {e}"
    return result

In [11]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model=model,
    tools=[travilySearch_tool, calculator_tool],
    system_prompt=SystemMessage("You are a Human resource assistant who identify relevant jobs for the user based on their skills and experience. Also calulate simple math expressions when needed."),
    # system_prompt="You are a helpful assistant and can also perform calculations as needed.",
)


user_input = "AI QA Engineer jobs  in Pune and also calculate '15 percent of 2000'."
# Pass a list of HumanMessage objects, not a dict
messages = [HumanMessage(content=user_input)]
response = agent.invoke({"messages": messages})
print(response)

{'messages': [HumanMessage(content="AI QA Engineer jobs  in Pune and also calculate '15 percent of 2000'.", additional_kwargs={}, response_metadata={}, id='f8d5918b-a1ed-4973-a658-00dc35ec588a'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'calculator_tool', 'arguments': '{"expression": "0.15 * 2000"}'}, '__gemini_function_call_thought_signatures__': {'449ac5d6-ee8f-483a-a2cf-8925897421bc': 'EvUCCvICAXLI2nzE0//UgqEt8+IqJThEkvG9koBjuBLnXXCxutHZ25ing9nf8ccOA565PzM7xMbJ6YvxAeuu/ozL1abvmsGzVD8C5/q65n5YvqpbLe3Y7FLneWERy9VqYNIRz2PYMbRTNk7WiBQagfAsNfhlWkVXm7hqM5ki/IK5VwM4JXuP0fYR0DCmpT+/Lt5UqB6iFZN75x8upxNcppRJDLM172JcT8rqun/b2W2WdMKjcPMyeUMI5X/y0xnZIn72F5bbGaQDHbN/X86P7VBZtAMhn+noQ+CGwY8tWRD1YoO5ttW/emdfM24Kf0jd5ZVpVC9mj0uFBhzJaFuC9Qxd2oeeGkp6rEar4l/FAdJmK8ptJ1n9/kVnt56h4cP4DTzgIiVZq9agso6Ro8O27G1gYp7xZgILXVeMNvqImfgiMtY5W/tD5JO83GaQIQUsEUcRSadAaDi3OLZW9x/J1qSUkDjeywdBql2B1DlwuUQ02zGBESu2+A=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash

In [12]:
response['messages'][-1].text

'Based on your interest in **AI QA Engineer** roles in **Pune**, here are several relevant job openings and the calculation you requested:\n\n### **AI QA Engineer Jobs in Pune**\n1.  **Software Engineer II (Quality) – Manual, Automation, AI & Cloud Testing** at **Mastercard**\n2.  **Senior QA Engineer – AI Rendering** at **Autodesk**\n3.  **Software QA Engineer – LLM Features** at **Autodesk** (Focused on Large Language Models)\n4.  **QA Engineer – AI Engineering** at **Agivant Technologies**\n5.  **Test Engineer – Medical Devices & AI** at **Codvo.ai** (Remote/Hybrid options often available)\n6.  **QA Data Science Engineer** at **Qualys**\n7.  **DevOps Engineer (with AI/ML focus)** at **Fission Labs**\n8.  **QA Automation Engineer** at **Centre for Computational Technologies (CCTech)**\n\n*Note: I recommend checking LinkedIn, Indeed, or the respective company career pages to apply directly, as these roles are highly sought after.*\n\n---\n\n### **Mathematical Calculation**\n**15 perce

In [13]:
for resp in agent.stream({"messages": messages},stream_mode='values'):
    resp['messages'][-1].pretty_print()

================================ Human Message =================================

AI QA Engineer jobs  in Pune and also calculate '15 percent of 2000'.
================================== Ai Message ==================================

[]
Tool Calls:
  tavily_search (a54a13bd-dd73-4917-a61b-866d6d6c2509)
 Call ID: a54a13bd-dd73-4917-a61b-866d6d6c2509
  Args:
    query: AI QA Engineer jobs in Pune
  calculator_tool (6e78dde4-5719-4e3f-9b59-8e8febeaa9b7)
 Call ID: 6e78dde4-5719-4e3f-9b59-8e8febeaa9b7
  Args:
    expression: 0.15 * 2000
================================== Ai Message ==================================

[]
Tool Calls:
  tavily_search (a54a13bd-dd73-4917-a61b-866d6d6c2509)
 Call ID: a54a13bd-dd73-4917-a61b-866d6d6c2509
  Args:
    query: AI QA Engineer jobs in Pune
  calculator_tool (6e78dde4-5719-4e3f-9b59-8e8febeaa9b7)
 Call ID: 6e78dde4-5719-4e3f-9b59-8e8febeaa9b7
  Args:
    expression: 0.15 * 2000
================================= Tool Message =============================

NameError: name 'agent' is not defined